In [ ]:
from chatterbot import ChatBot
from chatterbot.trainers import ListTrainer

## Remove WhatsApp Chat Metadata
In this section removes metadata (timestamps and usernames) from exported WhatsApp chat files.

In [ ]:
import re

def remove_chat_metadata(chat_export_file):
    # Updated pattern to handle formats like: 14/08/2025, 9:19 at night - monicaiyb: 
    date_time = r"\d+/\d+/\d+,\s\d+:\d+(?:\s(?:am|pm|at\s(?:night|morning|afternoon|noon1)))?"  
    dash_whitespace = r"\s-\s"  # " - "
    username = r"[^:]+"  # Everything until the colon
    metadata_end = r":\s"  # ": "
    pattern = date_time + dash_whitespace + username + metadata_end
    
    with open(chat_export_file, "r", encoding="utf-8") as corpus_file:
        content = corpus_file.read()
    
    cleaned_corpus = re.sub(pattern, "", content)
    return tuple(cleaned_corpus.split("\n"))

In [ ]:
def remove_non_message_text(export_text_lines):
    messages = export_text_lines[1:-1]
    filter_out_msgs = ("<Media omitted>",)
    return tuple((msg for msg in messages if msg not in filter_out_msgs))

In [ ]:
def clean_corpus(chat_export_file):
    message_corpus = remove_chat_metadata(chat_export_file)
    cleaned_corpus = remove_non_message_text(message_corpus)
    return cleaned_corpus

## Chatbot Data Preparation

These functions prepare WhatsApp chat data for chatbot training by extracting clean text messages.

In [ ]:
def prepare_for_chatbot(chat_export_file):
    """
    Complete pipeline to prepare WhatsApp chat for chatbot training.
    Returns clean text messages ready for use.
    """
    # Step 1: Remove metadata (timestamps and usernames)
    message_corpus = remove_chat_metadata(chat_export_file)
    
    # Step 2: Remove non-message text
    cleaned_corpus = remove_non_message_text(message_corpus)
    
    # Step 3: Apply text cleaning
    final_corpus = full_clean(
        cleaned_corpus,
        lowercase=True,          # Lowercase for consistency
        remove_punct=False,      # Keep punctuation (helps with context)
        remove_url=True,         # Remove URLs (not useful for chatbot)
        remove_num=False,        # Keep numbers (might be important)
        remove_whitespace=True,  # Clean whitespace
        remove_empty=True        # Remove empty messages
    )
    
    return final_corpus

def save_chatbot_data(messages, output_file="chatbot_training_data.txt"):
    """
    Save cleaned messages to a text file for chatbot training.
    Each message on a new line.
    """
    with open(output_file, "w", encoding="utf-8") as f:
        for message in messages:
            f.write(message + "\n")
    
    print(f"✓ Chatbot training data saved to: {output_file}")
    print(f"✓ Total messages: {len(messages)}")
    print(f"✓ Sample messages:")
    for i, msg in enumerate(messages[:5], 1):
        print(f"  {i}. {msg}")

def get_statistics(messages):
    """
    Display useful statistics about the cleaned data.
    """
    total_messages = len(messages)
    total_words = sum(len(msg.split()) for msg in messages)
    avg_words = total_words / total_messages if total_messages > 0 else 0
    
    print("=" * 50)
    print("CHATBOT TRAINING DATA STATISTICS")
    print("=" * 50)
    print(f"Total messages: {total_messages}")
    print(f"Total words: {total_words}")
    print(f"Average words per message: {avg_words:.2f}")
    print(f"Shortest message: {min(len(msg.split()) for msg in messages)} words")
    print(f"Longest message: {max(len(msg.split()) for msg in messages)} words")
    print("=" * 50)

## Complete Chatbot Data Preparation Pipeline

In [ ]:
# MAIN PIPELINE - Run this cell to prepare your chatbot training data

# 1. Prepare the data
print("Processing WhatsApp chat export...\n")
chatbot_messages = prepare_for_chatbot("WhatsAppChatwithTheresaSeryam.txt")

# 2. Show statistics
get_statistics(chatbot_messages)

# 3. Save to file
print("\nSaving cleaned data...\n")
save_chatbot_data(chatbot_messages, "chatbot_training_data.txt")

In [ ]:
# Preview the cleaned messages
print("First 10 cleaned messages for chatbot training:\n")
for i, msg in enumerate(chatbot_messages[:10], 1):
    print(f"{i}. {msg}")

In [ ]:
# Optional: Convert to list format for easy use in Python chatbot code
messages_list = list(chatbot_messages)
print(f"Data ready for chatbot! Total messages: {len(messages_list)}")
print(f"\nYou can now use 'messages_list' variable in your chatbot code.")

## Utility Cleaning Functions

These are helper functions used by the chatbot preparation pipeline.

In [ ]:
import string

def remove_punctuation(text_lines):
    """Remove punctuation from each message"""
    translator = str.maketrans('', '', string.punctuation)
    return tuple(msg.translate(translator) for msg in text_lines)

def convert_to_lowercase(text_lines):
    """Convert all messages to lowercase"""
    return tuple(msg.lower() for msg in text_lines)

def remove_extra_whitespace(text_lines):
    """Remove extra whitespace from messages"""
    return tuple(' '.join(msg.split()) for msg in text_lines)

def remove_empty_messages(text_lines):
    """Remove empty or whitespace-only messages"""
    return tuple(msg for msg in text_lines if msg.strip())

def remove_urls(text_lines):
    """Remove URLs from messages"""
    url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
    return tuple(re.sub(url_pattern, '', msg) for msg in text_lines)

def remove_numbers(text_lines):
    """Remove numbers from messages"""
    return tuple(re.sub(r'\d+', '', msg) for msg in text_lines)

def full_clean(text_lines, 
               lowercase=True, 
               remove_punct=True, 
               remove_url=True, 
               remove_num=False,
               remove_whitespace=True,
               remove_empty=True):
    """Apply multiple cleaning operations"""
    cleaned = text_lines
    
    if remove_url:
        cleaned = remove_urls(cleaned)
    
    if remove_punct:
        cleaned = remove_punctuation(cleaned)
    
    if remove_num:
        cleaned = remove_numbers(cleaned)
    
    if lowercase:
        cleaned = convert_to_lowercase(cleaned)
    
    if remove_whitespace:
        cleaned = remove_extra_whitespace(cleaned)
    
    if remove_empty:
        cleaned = remove_empty_messages(cleaned)
    
    return cleaned

## Usage Examples

In [ ]:
# Example usage - Method 1: Step by step
message_corpus = remove_chat_metadata("WhatsAppChatwithTheresaSeryam.txt")
cleaned_corpus = remove_non_message_text(message_corpus)
print("Messages after removing metadata:")
print(cleaned_corpus[:5])  # Show first 5 messages

In [ ]:
# Example usage - Method 2: Full cleaning pipeline
# Step 1: Remove metadata and non-message text
cleaned_corpus = clean_corpus("WhatsAppChatwithTheresaSeryam.txt")
print("Step 1 - After removing metadata:")
print(cleaned_corpus[:3])

# Step 2: Apply full cleaning
fully_cleaned = full_clean(cleaned_corpus)
print("\nStep 2 - After full cleaning:")
print(fully_cleaned[:3])

In [ ]:
# Custom cleaning - choose what to clean
custom_cleaned = full_clean(
    cleaned_corpus,
    lowercase=True,          # Convert to lowercase
    remove_punct=False,      # Keep punctuation
    remove_url=True,         # Remove URLs
    remove_num=False,        # Keep numbers
    remove_whitespace=True,  # Clean whitespace
    remove_empty=True        # Remove empty messages
)
print("Custom cleaned messages:")
print(custom_cleaned[:5])

In [ ]:
# Save cleaned data to file
output_file = "cleaned_chat.txt"
with open(output_file, "w", encoding="utf-8") as f:
    for message in fully_cleaned:
        f.write(message + "\n")
        
print(f"Cleaned data saved to {output_file}")
print(f"Total messages: {len(fully_cleaned)}")

In [ ]:
CORPUS_FILE = "cleaned_chat.txt"

chatbot = ChatBot("Chatpot")

In [ ]:
trainer = ListTrainer(chatbot)
cleaned_corpus = clean_corpus(CORPUS_FILE)
trainer.train(cleaned_corpus)

In [ ]:
exit_conditions = (":q", "quit", "exit")
while True:
    query = input("> ")
    if query in exit_conditions:
        break
    else:
        print(f"🪴 {chatbot.get_response(query)}")